In [0]:
# Workflow execution

from datetime import datetime

dbutils.widgets.text(
    "file_date",
    datetime.now().strftime("%Y%m%d")
)

file_date = dbutils.widgets.get("file_date")

print(f"Processing data for file_date = {file_date}")

In [0]:
# Manual execution

#file_date = "20260207"

#print(f"Processing data for file_date = {file_date}")

In [0]:
# Import required libraries

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
# If Gold tables already exist, remove data for the current file_date
# This allows the pipeline to be rerun safely for the same batch

gold_tables = [
    "dim_customers",
    "dim_products",
    "fact_orders",
    "fact_orders_rejected"
]

for table in gold_tables:
    table_name = f"`namastesql-databricks-catalog`.gold.{table}"

    if spark.catalog.tableExists(table_name):
        spark.sql(f"""
            DELETE FROM {table_name}
            WHERE file_date = '{file_date}'
        """)

### Create Customer Dimension (SCD Type 1)

In [0]:
# Read current file_date customer records from Silver layer
# Create a temporary view for SQL processing

(
    spark.table(
        "`namastesql-databricks-catalog`.silver.customers_unified"
    )
    .filter(col("file_date") == file_date)
    .createOrReplaceTempView("customers_src")
)

In [0]:
%sql

-- Create Customer Dimension table if it does not already exist

CREATE TABLE IF NOT EXISTS `namastesql-databricks-catalog`.gold.dim_customers
(
    customer_sk STRING,
    customer_id STRING,
    customer_name STRING,
    email STRING,
    mobile_clean STRING,
    city STRING,
    state STRING,
    zipcode STRING,
    signup_date STRING,
    store_id STRING,
    source STRING,
    file_date STRING,
    ingest_ts TIMESTAMP,
    last_update_ts TIMESTAMP
)
USING DELTA

In [0]:
%sql

MERGE INTO `namastesql-databricks-catalog`.gold.dim_customers AS tgt
USING
(
    SELECT
        sha2(mobile_clean, 256) AS customer_sk,
        customer_id,
        customer_name,
        email,
        mobile_clean,
        city,
        state,
        zipcode,
        signup_date,
        store_id,
        source,
        file_date,
        ingest_ts,

        sha2(
            concat_ws(
                '||',
                coalesce(customer_name, ''),
                coalesce(email, ''),
                coalesce(city, ''),
                coalesce(state, ''),
                coalesce(zipcode, ''),
                coalesce(signup_date, ''),
                coalesce(store_id, '')
            ),
            256
        ) AS change_hash

    FROM customers_src
) src

ON tgt.customer_sk = src.customer_sk

WHEN MATCHED
AND
sha2(
    concat_ws(
        '||',
        coalesce(tgt.customer_name, ''),
        coalesce(tgt.email, ''),
        coalesce(tgt.city, ''),
        coalesce(tgt.state, ''),
        coalesce(tgt.zipcode, ''),
        coalesce(tgt.signup_date, ''),
        coalesce(tgt.store_id, '')
    ),
    256
)
<>
src.change_hash

THEN UPDATE SET
    tgt.customer_id      = src.customer_id,
    tgt.customer_name    = src.customer_name,
    tgt.email            = src.email,
    tgt.city             = src.city,
    tgt.state            = src.state,
    tgt.zipcode          = src.zipcode,
    tgt.signup_date      = src.signup_date,
    tgt.store_id         = src.store_id,
    tgt.source           = src.source,
    tgt.file_date        = src.file_date,
    tgt.last_update_ts   = current_timestamp()

WHEN NOT MATCHED
THEN INSERT
(
    customer_sk,
    customer_id,
    customer_name,
    email,
    mobile_clean,
    city,
    state,
    zipcode,
    signup_date,
    store_id,
    source,
    file_date,
    ingest_ts,
    last_update_ts
)
VALUES
(
    src.customer_sk,
    src.customer_id,
    src.customer_name,
    src.email,
    src.mobile_clean,
    src.city,
    src.state,
    src.zipcode,
    src.signup_date,
    src.store_id,
    src.source,
    src.file_date,
    src.ingest_ts,
    src.ingest_ts
)

### Create Product Dimension (SCD Type 2)

In [0]:
# Read current file_date product records from Silver layer
# Create a temporary view for SQL processing

(
    spark.table(
        "`namastesql-databricks-catalog`.silver.products_unified"
    )
    .filter(col("file_date") == file_date)
    .createOrReplaceTempView("products_src")
)

In [0]:
%sql

-- Create Product Dimension table if it does not already exist

CREATE TABLE IF NOT EXISTS `namastesql-databricks-catalog`.gold.dim_products
(
    product_sk STRING,
    sku_id STRING,
    product_name STRING,
    category STRING,
    brand STRING,
    price DOUBLE,
    tax_rate DOUBLE,
    currency STRING,
    is_active STRING,
    effective_from DATE,
    effective_to DATE,
    is_current STRING,
    source STRING,
    file_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW products_stage AS
SELECT
    sha2(concat_ws('||', sku_id, file_date), 256) AS product_sk,
    sku_id,
    product_name,
    category,
    brand,
    price,
    tax_rate,
    currency,
    is_active,
    to_date(file_date, 'yyyyMMdd') AS effective_from,
    CAST('9999-12-31' AS DATE) AS effective_to,
    'Y' AS is_current,
    source,
    file_date,
    ingest_ts,
    sha2(
        concat_ws(
            '||',
            coalesce(product_name, ''),
            coalesce(category, ''),
            coalesce(brand, ''),
            coalesce(cast(price as string), ''),
            coalesce(cast(tax_rate as string), ''),
            coalesce(currency, ''),
            coalesce(is_active, '')
        ),
        256
    ) AS change_hash
FROM products_src

In [0]:
%sql

MERGE INTO `namastesql-databricks-catalog`.gold.dim_products tgt
USING products_stage src
ON tgt.sku_id = src.sku_id
AND tgt.is_current = 'Y'

WHEN MATCHED
AND
sha2(
    concat_ws(
        '||',
        coalesce(tgt.product_name, ''),
        coalesce(tgt.category, ''),
        coalesce(tgt.brand, ''),
        coalesce(cast(tgt.price as string), ''),
        coalesce(cast(tgt.tax_rate as string), ''),
        coalesce(tgt.currency, ''),
        coalesce(tgt.is_active, '')
    ),
    256
)
<> src.change_hash

THEN UPDATE SET
    tgt.effective_to = date_sub(src.effective_from, 1),
    tgt.is_current = 'N'

In [0]:
%sql

INSERT INTO `namastesql-databricks-catalog`.gold.dim_products
SELECT
    src.product_sk,
    src.sku_id,
    src.product_name,
    src.category,
    src.brand,
    src.price,
    src.tax_rate,
    src.currency,
    src.is_active,
    src.effective_from,
    src.effective_to,
    src.is_current,
    src.source,
    src.file_date,
    src.ingest_ts
FROM products_stage src
LEFT JOIN `namastesql-databricks-catalog`.gold.dim_products tgt
    ON src.sku_id = tgt.sku_id
    AND tgt.is_current = 'Y'
WHERE tgt.sku_id IS NULL

### Create Orders Fact Table

In [0]:
# Read current file_date orders from Silver layer
# Create a temporary view for SQL processing

(
    spark.table(
        "`namastesql-databricks-catalog`.silver.orders_unified"
    )
    .filter(col("file_date") == file_date)
    .createOrReplaceTempView("orders_src")
)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS `namastesql-databricks-catalog`.gold.fact_orders
(
    order_id STRING,
    order_ts TIMESTAMP,
    customer_sk STRING,
    product_sk STRING,
    quantity INT,
    list_price DOUBLE,
    discount DOUBLE,
    amount DOUBLE,
    payment_mode STRING,
    store_num STRING,
    source STRING,
    file_date STRING,
    ingest_ts TIMESTAMP
)
USING DELTA

In [0]:
%sql

CREATE TABLE IF NOT EXISTS `namastesql-databricks-catalog`.gold.fact_orders_rejected
(
    order_id STRING,
    order_ts TIMESTAMP,
    customer_id STRING,
    sku_id STRING,
    quantity INT,
    list_price DOUBLE,
    discount DOUBLE,
    amount DOUBLE,
    payment_mode STRING,
    store_num STRING,
    source STRING,
    file_date STRING,
    ingest_ts TIMESTAMP,
    reject_reason STRING
)
USING DELTA

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW orders_stage AS
SELECT
    o.order_id,
    o.order_ts,
    c.customer_sk,
    p.product_sk,
    o.customer_id,
    o.sku_id,
    o.quantity,
    o.list_price,
    o.discount,
    o.amount,
    o.payment_mode,
    o.store_num,
    o.source,
    o.file_date,
    o.ingest_ts
FROM orders_src o

LEFT JOIN `namastesql-databricks-catalog`.gold.dim_customers c
    ON o.customer_id = c.customer_id

LEFT JOIN `namastesql-databricks-catalog`.gold.dim_products p
    ON o.sku_id = p.sku_id
    AND to_date(o.file_date, 'yyyyMMdd')
        BETWEEN p.effective_from
        AND p.effective_to

In [0]:
%sql

INSERT INTO `namastesql-databricks-catalog`.gold.fact_orders_rejected
SELECT
    order_id,
    order_ts,
    customer_id,
    sku_id,
    quantity,
    list_price,
    discount,
    amount,
    payment_mode,
    store_num,
    source,
    file_date,
    ingest_ts,
    CASE
        WHEN customer_sk IS NULL
             AND product_sk IS NULL
        THEN 'Customer and Product not found'

        WHEN customer_sk IS NULL
        THEN 'Customer not found'

        WHEN product_sk IS NULL
        THEN 'Product not found'
    END AS reject_reason
FROM orders_stage
WHERE customer_sk IS NULL
   OR product_sk IS NULL

In [0]:
%sql

INSERT INTO `namastesql-databricks-catalog`.gold.fact_orders
SELECT
    order_id,
    order_ts,
    customer_sk,
    product_sk,
    quantity,
    list_price,
    discount,
    amount,
    payment_mode,
    store_num,
    source,
    file_date,
    ingest_ts
FROM orders_stage
WHERE customer_sk IS NOT NULL
  AND product_sk IS NOT NULL